# Forward 12-Month Drawdown Prediction — End-to-End Deliverable

**Course**: Machine Learning II, MSADS, University of Chicago, Spring 2026  
**Team**: Nick Dhaliwal, Jared Maksoud, Nicholas Mikhail, Yung Chyi Yang

This notebook is the single runnable artifact for the project. It loads the
WRDS parquet files, builds the (gvkey, fyear) anchor panel with the corrected
forward window, computes targets and features, fits the three baselines
(vol-only / Ridge / XGBoost), trains the five fusion configurations across
three seeds each, and emits the full §8.9 metric set plus the slide-deck
visuals.

Spec source of truth: `PROJECT_BRIEF.md` and `reports/progress_report_2026-05-19.md`.
All heavy lifting lives in `src/`. The notebook is mostly orchestration so the
team can scan it linearly without scrolling through training code.


## 0. Setup and configuration

Knobs to edit before running:
- `DATA_DIR` — path to your local copy of the WRDS parquets.
- `N_SEEDS` — seeds per fusion config. Default 3 gives mean ± std headlines.
- `MAX_EPOCHS` — 50 per brief; drop to 10 for a fast pass.
- `SMOKE_TEST` — `True` subsamples to ~2k anchors and 3 epochs (~5 min).


In [ ]:
import os, sys, json, time, warnings, pickle
from pathlib import Path

# Make src/ importable when the notebook runs from notebooks/
REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))
warnings.filterwarnings('ignore')

# === edit these ===
DATA_DIR    = Path('/Users/nd/Documents/UChicago MSADS 2025-26/Machine Learning II/Final Project/MLII Project Data')
OUT_DIR     = REPO_ROOT / 'reports' / 'outputs'
FIG_DIR     = REPO_ROOT / 'reports' / 'figures'
N_SEEDS     = 3
MAX_EPOCHS  = 50
SMOKE_TEST  = False

OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)
print(f'Repo root : {REPO_ROOT}')
print(f'Data dir  : {DATA_DIR}')
print(f'Smoke test: {SMOKE_TEST}')


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.data.anchors import build_anchors, summarize_anchors
from src.data.targets import build_targets, summarize_targets
from src.data.splits import TRAIN_YEARS, VAL_YEARS, TEST_YEARS, split_masks
from src.tabular.ratios import (
    RATIO_COLS, compute_ratios, FinancialFeatureScaler,
    build_financial_matrix, flatten_for_baselines, flat_feature_names,
)
from src.price.features import (
    PRICE_COLS, build_price_features, PriceFeatureScaler,
)
from src.eval.metrics import evaluate, results_table, HEADLINE_COLS, APPENDIX_COLS
from src.fusion.train import run_multi_seed, TrainConfig
from src.fusion.model import CONFIG_NAMES
from src.eval.plots import (
    gics_sector_breakdown, calibration_scatter, named_firm_trajectories,
    reliability_diagram, feature_importance,
)
print('Modules loaded.')


## 1. Load raw data

Six WRDS parquets per `PROJECT_BRIEF.md` §5. The msf file is redundant and
is loaded only if `LOAD_MSF` is `True`.


In [ ]:
t0 = time.time()
funda     = pd.read_parquet(DATA_DIR / 'compustat_funda.parquet')
company   = pd.read_parquet(DATA_DIR / 'compustat_company.parquet')
bklabels  = pd.read_parquet(DATA_DIR / 'compustat_bklabels.parquet')
linktable = pd.read_parquet(DATA_DIR / 'crsp_linktable.parquet')
dsf       = pd.read_parquet(DATA_DIR / 'crsp_dsf.parquet')

funda['datadate']        = pd.to_datetime(funda['datadate'])
linktable['linkdt']      = pd.to_datetime(linktable['linkdt'])
linktable['linkenddt']   = pd.to_datetime(linktable['linkenddt'])
dsf['date']              = pd.to_datetime(dsf['date'])
company['dldte']         = pd.to_datetime(company['dldte'])

print(f'funda    : {funda.shape}')
print(f'company  : {company.shape}')
print(f'bklabels : {bklabels.shape}')
print(f'linktable: {linktable.shape}')
print(f'dsf      : {dsf.shape}')
print(f'load wall-time: {time.time() - t0:.1f}s')


## 2. Build the anchor panel

Per brief §8.1: `anchor_date = datadate + 90 days`, link permno via
`linkdt <= datadate <= linkenddt` (prefer `linkprim='P'` on ties), require
5 prior fiscal years of funda, restrict to fyear 2003..2024.


In [ ]:
anchors = build_anchors(funda, linktable, min_fyear=2003, max_fyear=2024,
                        require_history=True)
summarize_anchors(anchors)


## 3. Build the forward 12-month drawdown target

365-calendar-day forward window with a 60-trading-day minimum. Delisting
rules (brief §8.2): dlrsn 02/03/04 = full loss, dlrsn 01/07 = terminal
value (price series simply ends). This is the corrected target from
`src/data/targets.py`; the original 02 notebook used an unbounded
252-row window that inflated drawdowns badly.


In [ ]:
targets = build_targets(anchors, dsf, company)
summarize_targets(targets)


In [ ]:
# Smoke-test subsample for fast iteration (keeps the anchor structure intact)
if SMOKE_TEST:
    targets = targets.sample(n=min(2000, len(targets)), random_state=0).reset_index(drop=True)
    print(f'SMOKE_TEST: subsampled to {len(targets)} anchors')


## 4. Compute the 18 financial ratios

Brief §7 locked set. Ratios are computed on the full funda panel then
winsorized at the [1%, 99%] bounds and z-scored using TRAIN-FOLD-ONLY
statistics. Fixes the all-data scaling leakage in the original baseline.


In [ ]:
ratios_full = compute_ratios(funda)[['gvkey', 'fyear'] + RATIO_COLS].copy()
train_year_mask = ratios_full['fyear'].isin(TRAIN_YEARS).to_numpy()
scaler = FinancialFeatureScaler(low=0.01, high=0.99)
ratios_scaled = scaler.fit_transform(ratios_full, train_year_mask)
print(f'Scaled ratios: {ratios_scaled.shape}')


## 5. Build the financial (5,18) matrix and the price (7,) vector

`build_financial_matrix` stacks five lagged fyears per anchor; the leakage
assertion inside `build_price_features` enforces window-end < anchor_date.


In [ ]:
X_fin_3d, valid_mask = build_financial_matrix(targets, ratios_scaled, require_full_window=True)
targets = targets.loc[valid_mask].reset_index(drop=True)
X_fin_3d = X_fin_3d[valid_mask]
print(f'Financial matrix: {X_fin_3d.shape}    (kept {valid_mask.sum():,} / {len(valid_mask):,} anchors)')


In [ ]:
X_price_raw = build_price_features(targets, dsf)
good_price = np.all(np.isfinite(X_price_raw), axis=1)
targets = targets.loc[good_price].reset_index(drop=True)
X_fin_3d = X_fin_3d[good_price]
X_price_raw = X_price_raw[good_price]
print(f'Price matrix    : {X_price_raw.shape}    (dropped {(~good_price).sum():,} for missing price features)')


## 6. Time-blocked split + per-fyear price scaling on TRAIN ONLY

Train 2003..2017, Val 2018..2019, Test 2020..2023. Price-feature scaler
fits on training rows only and applies frozen stats to all folds.


In [ ]:
fyears = targets['fyear'].to_numpy()
masks = split_masks(fyears)
print('Anchors per fold:', {k: int(v.sum()) for k, v in masks.items()})

price_scaler = PriceFeatureScaler().fit(X_price_raw, fyears, masks['train'])
X_price = price_scaler.transform(X_price_raw, fyears)
print(f'X_price scaled  : {X_price.shape}')


In [ ]:
# Flatten the (5,18) matrix to (90,) and concat the 7 price features for the
# baseline models. Keep the 3-D version for the LSTM and the fusion models.
X_fin_flat = flatten_for_baselines(X_fin_3d)
X_full = np.concatenate([X_fin_flat, X_price], axis=1)
all_cols = flat_feature_names() + PRICE_COLS
print(f'Flat baseline X : {X_full.shape}')
y = targets['fwd_12m_max_dd'].to_numpy()
print(f'y               : {y.shape}   mean {y.mean():.3f}   median {np.median(y):.3f}')


## 7. Baselines (brief §8.6)

Three baselines on the same fold structure:
- **Vol-only**: predict `dd = a * trailing_vol + b`, fit on train.
- **Ridge**: 97-d flattened features with RidgeCV alpha sweep.
- **XGBoost**: same 97-d input, small hyperparameter search with early
  stopping on val MAE.


In [ ]:
from sklearn.linear_model import LinearRegression, RidgeCV
import xgboost as xgb

yrs = fyears
y_tr = y[masks['train']]; y_va = y[masks['val']]; y_te = y[masks['test']]
yrs_te = yrs[masks['test']]

# Trailing vol is column index PRICE_COLS.index('pf_vol') in X_full's last 7 cols
vol_idx = X_full.shape[1] - len(PRICE_COLS) + PRICE_COLS.index('pf_vol')
vol = X_full[:, vol_idx]
vol_tr = vol[masks['train']].reshape(-1, 1)
vol_te = vol[masks['test']].reshape(-1, 1)

vol_model = LinearRegression().fit(vol_tr, y_tr)
y_vol = vol_model.predict(vol_te)
vol_row = evaluate('vol_only', y_te, y_vol, yrs_te, verbose=True)


In [ ]:
ridge = RidgeCV(alphas=[0.01, 0.1, 1.0, 10.0, 100.0, 1000.0],
                scoring='neg_mean_absolute_error').fit(X_full[masks['train']], y_tr)
y_ridge = ridge.predict(X_full[masks['test']])
ridge_row = evaluate(f'ridge_a{ridge.alpha_:g}', y_te, y_ridge, yrs_te, verbose=True)


In [ ]:
# Small XGBoost search. Train on train, early-stop on val MAE.
param_grid = [
    dict(n_estimators=200, max_depth=3, learning_rate=0.05),
    dict(n_estimators=500, max_depth=3, learning_rate=0.05),
    dict(n_estimators=500, max_depth=5, learning_rate=0.05),
    dict(n_estimators=500, max_depth=5, learning_rate=0.1),
    dict(n_estimators=500, max_depth=7, learning_rate=0.05),
    dict(n_estimators=500, max_depth=7, learning_rate=0.1),
]
best = (None, None, float('inf'))
for p in param_grid:
    m = xgb.XGBRegressor(objective='reg:absoluteerror',
                         eval_metric='mae',
                         early_stopping_rounds=30,
                         tree_method='hist', verbosity=0, **p)
    m.fit(X_full[masks['train']], y_tr,
          eval_set=[(X_full[masks['val']], y_va)], verbose=False)
    val_mae = float(np.mean(np.abs(m.predict(X_full[masks['val']]) - y_va)))
    if val_mae < best[2]:
        best = (m, p, val_mae)
xgb_model, xgb_params, xgb_val_mae = best
print(f'XGBoost best val MAE: {xgb_val_mae:.4f}, params: {xgb_params}')
y_xgb = xgb_model.predict(X_full[masks['test']])
xgb_row = evaluate('xgboost', y_te, y_xgb, yrs_te, verbose=True)


## 8. Fusion model — five configurations × N_SEEDS seeds

Configurations (brief §8.7, §8.8 + TA cross-attention ablation):
1. `lstm_fusion` — LSTM financial encoder, full fusion (headline candidate).
2. `mlp_fusion`  — MLP-flatten financial encoder, full fusion.
3. `fin_only`    — financials-only (drop price branch).
4. `price_only`  — price-only (drop financial branch).
5. `cross_attn_fusion` — single attention head, LSTM hidden states ⇄ price embedding.


In [ ]:
epochs = 3 if SMOKE_TEST else MAX_EPOCHS
seeds = list(range(N_SEEDS))
train_cfg = TrainConfig(epochs=epochs, verbose=True)

tr, va, te = masks['train'], masks['val'], masks['test']
kwargs = dict(
    X_fin_tr=X_fin_3d[tr], X_price_tr=X_price[tr], y_tr=y_tr,
    X_fin_va=X_fin_3d[va], X_price_va=X_price[va], y_va=y_va,
    X_fin_te=X_fin_3d[te], X_price_te=X_price[te], y_te=y_te,
    yrs_te=yrs_te,
    seeds=seeds, train_cfg=train_cfg,
)
fusion_runs = {}
for config in CONFIG_NAMES:
    print(f'\n========== {config} ==========')
    fusion_runs[config] = run_multi_seed(config, **kwargs)


## 9. Headline results table

All on the corrected target and the same metric definitions. Means across
seeds shown for fusion configs; baselines have a single deterministic run.


In [ ]:
rows = [vol_row, ridge_row, xgb_row]
for cfg, summary in fusion_runs.items():
    # Pick the best-val-seed row for display; the appendix table reports mean±std.
    best_row = next(r for r in summary['seed_rows'] if r['best_val_mae'] == min(s['best_val_mae'] for s in summary['seed_rows']))
    best_row = dict(best_row)
    best_row['model'] = cfg
    rows.append(best_row)

headline = results_table(rows, cols=HEADLINE_COLS)
headline_seedmean = pd.DataFrame([
    {'model': cfg,
     'mae': s['mae_mean'], 'mae_std': s['mae_std'],
     'pr_auc_30': s['pr_auc_30_mean'], 'pr_auc_30_std': s['pr_auc_30_std'],
     'spearman_within_year': s['spearman_within_year_mean'],
     'top_decile_prec_within_year': s['top_decile_prec_within_year_mean']}
    for cfg, s in fusion_runs.items()
]).set_index('model')

print('\n=== Headline (best seed for fusion configs) ===')
print(headline.round(4).to_string())

print('\n=== Fusion seed mean ± std ===')
print(headline_seedmean.round(4).to_string())

headline.to_csv(OUT_DIR / 'results_headline.csv')
headline_seedmean.to_csv(OUT_DIR / 'results_fusion_seedmean.csv')


## 10. Interpretability — GICS sector breakdown on the COVID anchor

Brief §8.10 / methodology §5: on fyear 2018 anchors (target window covers
March 2019 → March 2020), the model should flag airlines, hospitality,
cruise lines, and retail as elevated risk before COVID arrives.


In [ ]:
# Pick the headline model: best-val LSTM fusion seed.
headline_cfg = 'lstm_fusion'
headline_preds_test = fusion_runs[headline_cfg]['best_preds_test']

test_df = targets.loc[masks['test']].reset_index(drop=True).copy()
test_df['y_pred'] = headline_preds_test
test_df['y_true'] = y_te

# To get fyear 2018 (COVID anchor) we need val-fold predictions too
best_seed = fusion_runs[headline_cfg]['best_seed']
# Recompute predictions for val fold using the saved best state dict
import torch
from src.fusion.model import build_model
from src.fusion.train import pick_device
device = pick_device('auto')
model_h = build_model(headline_cfg).to(device)
model_h.load_state_dict({k: v.to(device) for k, v in fusion_runs[headline_cfg]['best_state_dict'].items()})
model_h.eval()
with torch.no_grad():
    val_preds = model_h(
        torch.as_tensor(X_fin_3d[masks['val']], dtype=torch.float32).to(device),
        torch.as_tensor(X_price[masks['val']], dtype=torch.float32).to(device),
    ).cpu().numpy()

val_df = targets.loc[masks['val']].reset_index(drop=True).copy()
val_df['y_pred'] = val_preds
val_df['y_true'] = y[masks['val']]

covid_df = val_df[val_df['fyear'] == 2018].merge(
    company[['gvkey', 'gsector', 'conm']].drop_duplicates('gvkey'),
    on='gvkey', how='left'
).dropna(subset=['gsector'])
print(f'COVID anchor (fyear 2018): {len(covid_df):,} firms with GICS sector')


In [ ]:
fig, ax = gics_sector_breakdown(covid_df)
fig.savefig(FIG_DIR / 'gics_sector_breakdown_covid.png', dpi=150, bbox_inches='tight')
plt.show()


## 11. Calibration scatter + reliability diagram


In [ ]:
fig, ax = calibration_scatter(test_df['y_true'].to_numpy(),
                              test_df['y_pred'].to_numpy(),
                              test_df['fyear'].to_numpy())
fig.savefig(FIG_DIR / 'calibration_scatter_test.png', dpi=150, bbox_inches='tight')
plt.show()

fig, ax = reliability_diagram(test_df['y_true'].to_numpy(), test_df['y_pred'].to_numpy(),
                              threshold=-0.30, n_bins=10)
fig.savefig(FIG_DIR / 'reliability_dd30_test.png', dpi=150, bbox_inches='tight')
plt.show()


## 12. Named-firm trajectories

Three distressed firms (American Airlines, Carnival, Bed Bath & Beyond)
and three healthy peers (Procter & Gamble, Microsoft, Johnson & Johnson)
across the test fyears.


In [ ]:
test_named = test_df.merge(
    company[['gvkey', 'conm', 'tic']].drop_duplicates('gvkey') if 'tic' in company.columns else company[['gvkey', 'conm']].drop_duplicates('gvkey'),
    on='gvkey', how='left'
)
wanted = ['AMERICAN AIRLINES GROUP INC', 'CARNIVAL CORP/PLC', 'BED BATH & BEYOND INC',
          'PROCTER & GAMBLE CO', 'MICROSOFT CORP', 'JOHNSON & JOHNSON']
found = [c for c in wanted if (test_named['conm'].str.upper() == c).any()]
print('Firms found:', found)
plot_df = test_named[test_named['conm'].str.upper().isin(found)].copy()
plot_df['conm'] = plot_df['conm'].str.title()
fig, axes = named_firm_trajectories(plot_df, firms=plot_df['conm'].unique().tolist())
fig.savefig(FIG_DIR / 'named_firm_trajectories.png', dpi=150, bbox_inches='tight')
plt.show()


## 13. Bankruptcy cross-check

Secondary validation using the 387-firm Chapter 7/11 set in
`compustat_bklabels`. The headline finding from the prior run was that only
~2 of 387 bankruptcies appear in the anchor set — the rest delisted before
a 12-month forward window could be computed. This is the survivorship-bias
story the instructor flagged.


In [ ]:
bk = bklabels[['gvkey', 'dldte']].copy()
bk['dldte'] = pd.to_datetime(bk['dldte'])
# A bankruptcy is 'covered' by an anchor if any anchor's forward window contains the dldte
anchors_full_test = targets.loc[masks['test']].copy()
anchors_full_test['win_start'] = pd.to_datetime(anchors_full_test['anchor_date'])
anchors_full_test['win_end'] = anchors_full_test['win_start'] + pd.Timedelta(days=365)
covered = []
for _, row in bk.iterrows():
    m = (anchors_full_test['gvkey'] == row['gvkey']) & \
        (anchors_full_test['win_start'] <= row['dldte']) & \
        (row['dldte'] <= anchors_full_test['win_end'])
    if m.any():
        covered.append(row['gvkey'])
n_cov = len(set(covered))
print(f'Bankruptcies covered by a test-fold anchor: {n_cov} / {len(bk)} ({n_cov / len(bk):.1%})')


## 14. -50% appendix table

Brief locks -30% as the headline binary threshold but the realized base
rate at -30% is ~55% on the CRSP small-cap-heavy universe. -50% drops the
base rate to ~26%, closer to the brief's expected 10-20% tail mass.
Reported as appendix per the locked decision.


In [ ]:
appendix = results_table(rows, cols=APPENDIX_COLS)
print(appendix.round(4).to_string())
appendix.to_csv(OUT_DIR / 'results_appendix_dd50.csv')


## 15. Save trained checkpoints + bundle for handoff

Pickled per-config best state dict, prediction arrays, and the full results
manifest go under `reports/outputs/` so the team can rerun the interpretability
section without retraining.


In [ ]:
import torch
for cfg, s in fusion_runs.items():
    torch.save(s['best_state_dict'], OUT_DIR / f'weights_{cfg}_seed{s["best_seed"]}.pt')
manifest = {
    'configs': list(fusion_runs.keys()),
    'n_seeds': N_SEEDS,
    'n_anchors': int(len(targets)),
    'fyear_min': int(targets['fyear'].min()),
    'fyear_max': int(targets['fyear'].max()),
    'base_rate_30': float((y <= -0.30).mean()),
    'base_rate_50': float((y <= -0.50).mean()),
    'bankruptcies_covered': n_cov,
    'bankruptcies_total': int(len(bk)),
    'best_seed_per_config': {cfg: s['best_seed'] for cfg, s in fusion_runs.items()},
}
with open(OUT_DIR / 'manifest.json', 'w') as f:
    json.dump(manifest, f, indent=2)
print(json.dumps(manifest, indent=2))


## 16. Recap

Single end-to-end notebook delivering the brief's headline:
- corrected forward-window target via `targets.py`
- leak-free train-fold-only feature scaling
- 97-d baselines (vol-only / Ridge / XGBoost) for comparison floors
- 5 fusion configurations × 3 seeds (mean ± std), including cross-attention
  ablation per TA priority
- within-year Spearman and top-decile precision (not pooled)
- GICS COVID-anchor visual, calibration, reliability, named-firm trajectories
- bankruptcy cross-check capturing the survivorship-bias story
- -30% headline + -50% appendix

Writeup paragraphs for the four TA-flagged items live in
`reports/writeup_appendix.md`.
